In [1]:
# Phase 1: Problem Framing
business_question = "What drives monthly donations, and how can we predict next month donations?"
success_metrics = ["Lower MAE", "Lower RMSE", "Higher R2 on test data"]
approach = {
    "causal_model": "Explanatory OLS regression (relationship estimates, not proof of causality)",
    "predictive_model": "Random Forest regression (best out-of-sample prediction)"
}

print("Business Question:", business_question)
print("Success Metrics:", success_metrics)
print("Approach:", approach)

Business Question: What drives monthly donations, and how can we predict next month donations?
Success Metrics: ['Lower MAE', 'Lower RMSE', 'Higher R2 on test data']
Approach: {'causal_model': 'Explanatory OLS regression (relationship estimates, not proof of causality)', 'predictive_model': 'Random Forest regression (best out-of-sample prediction)'}


In [2]:
# Phase 2: Data Acquisition and Preparation
import ast
import json
import pandas as pd

file_path = "../datasets/public_impact_snapshots.csv"
df = pd.read_csv(file_path)

payload = df["metric_payload_json"].apply(ast.literal_eval).apply(pd.Series)
df2 = pd.concat([df.drop(columns=["metric_payload_json"]), payload], axis=1)

df2["snapshot_date"] = pd.to_datetime(df2["snapshot_date"])
df2["published_at"] = pd.to_datetime(df2["published_at"], errors="coerce")
df2["month_number"] = df2["snapshot_date"].dt.month
df2["year"] = df2["snapshot_date"].dt.year

df_model = df2[[
    "avg_health_score",
    "avg_education_progress",
    "total_residents",
    "month_number",
    "year",
    "donations_total_for_month"
]].dropna()

print("Rows and Columns:", df_model.shape)
print(df_model.head())

Rows and Columns: (50, 6)
   avg_health_score  avg_education_progress  total_residents  month_number  \
0              3.03                   33.90               60             1   
1              3.13                   51.05               60             2   
2              3.16                   60.57               60             3   
3              3.20                   69.15               60             4   
4              3.20                   78.85               60             5   

   year  donations_total_for_month  
0  2023                    1379.92  
1  2023                    2065.15  
2  2023                    9577.83  
3  2023                    5401.47  
4  2023                    4862.09  


In [3]:
# Phase 3: Exploration
print("Summary Statistics")
print(df_model.describe())

print("\nCorrelations with donations_total_for_month")
print(df_model.corr(numeric_only=True)["donations_total_for_month"].sort_values(ascending=False))

Summary Statistics
       avg_health_score  avg_education_progress  total_residents  \
count          50.00000               50.000000             50.0   
mean            2.45060               59.499000             60.0   
std             1.40025               35.194204              0.0   
min             0.00000                0.000000             60.0   
25%             3.04250               38.187500             60.0   
50%             3.13500               77.485000             60.0   
75%             3.20000               81.717500             60.0   
max             3.94000              100.000000             60.0   

       month_number         year  donations_total_for_month  
count     50.000000    50.000000                  50.000000  
mean       6.300000  2024.600000                4814.490800  
std        3.558548     1.212183                4512.487236  
min        1.000000  2023.000000                   0.000000  
25%        3.000000  2024.000000                 902.43750

In [4]:
# Phase 4: Modeling (Causal + Predictive)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm

X = df_model.drop(columns=["donations_total_for_month"])
y = df_model["donations_total_for_month"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Causal / explanatory model (OLS)
X_train_ols = sm.add_constant(X_train)
causal_model = sm.OLS(y_train, X_train_ols).fit()

# Predictive model
predictive_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])
predictive_model.fit(X_train, y_train)

print("Causal model and predictive model trained.")

Causal model and predictive model trained.


In [5]:
# Phase 5: Evaluation and Selection
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = predictive_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Predictive Model Metrics")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

# Fairness check (simple): no demographic group columns available
print("Fairness check: demographic group fields are not present in this dataset.")

print("\nCausal Model (OLS) summary:")
print(causal_model.summary())

Predictive Model Metrics
MAE: 2062.78
RMSE: 2709.36
R2: 0.3895
Fairness check: demographic group fields are not present in this dataset.

Causal Model (OLS) summary:
                                OLS Regression Results                               
Dep. Variable:     donations_total_for_month   R-squared:                       0.584
Model:                                   OLS   Adj. R-squared:                  0.537
Method:                        Least Squares   F-statistic:                     12.30
Date:                       Tue, 07 Apr 2026   Prob (F-statistic):           2.40e-06
Time:                               22:05:18   Log-Likelihood:                -376.99
No. Observations:                         40   AIC:                             764.0
Df Residuals:                             35   BIC:                             772.4
Df Model:                                  4                                         
Covariance Type:                   nonrobust                

In [6]:
# Phase 6: Feature Selection / Impact
# Explanatory impact (OLS coefficients)
coef_table = causal_model.params.sort_values(key=lambda s: s.abs(), ascending=False)
print("Top explanatory coefficients (absolute size):")
print(coef_table)

# Predictive impact (Random Forest feature importances)
rf_model = predictive_model.named_steps["model"]
importance_table = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop predictive feature importances:")
print(importance_table)

print("\nRecommended decisions based on top features:")
print("1) Prioritize programs that improve education progress.")
print("2) Track health score trends monthly and intervene early if scores drop.")
print("3) Use seasonal month patterns for campaign timing.")

Top explanatory coefficients (absolute size):
total_residents          -19246.983147
avg_health_score           4083.583650
month_number                728.832332
year                        567.528788
avg_education_progress      -67.336047
dtype: float64

Top predictive feature importances:
month_number              0.376806
avg_education_progress    0.317571
avg_health_score          0.170878
year                      0.134745
total_residents           0.000000
dtype: float64

Recommended decisions based on top features:
1) Prioritize programs that improve education progress.
2) Track health score trends monthly and intervene early if scores drop.
3) Use seasonal month patterns for campaign timing.


In [7]:
# Phase 7: Deployment (simple artifact + sample payload)
import os
import joblib

os.makedirs("../artifacts", exist_ok=True)

artifact_path = "../artifacts/public_impact_snapshots_production_model.joblib"
joblib.dump(predictive_model, artifact_path)

sample_payload = {
    "avg_health_score": 3.2,
    "avg_education_progress": 78.0,
    "total_residents": 60,
    "month_number": 4,
    "year": 2026
}

sample_df = pd.DataFrame([sample_payload])
sample_prediction = predictive_model.predict(sample_df)[0]

with open("../artifacts/public_impact_snapshots_sample_payload.json", "w", encoding="utf-8") as f:
    json.dump(sample_payload, f, indent=2)

print("Saved model to:", artifact_path)
print("Sample predicted monthly donations:", round(float(sample_prediction), 2))

Saved model to: ../artifacts/public_impact_snapshots_production_model.joblib
Sample predicted monthly donations: 4563.41
